# Arabic Text Error Detection, Correction & Morphological Analysis

**v2 — spelling and grammar detection/correction now run on the whole text at
once, using a sequence-to-sequence transformer**, replacing the earlier
word-by-word/rule-based version. Normalization, stemming, and lemma analysis
are unchanged.

Pipeline:

1. **Verify** the input is actually Arabic
2. **Tokenize** the text
3. **Correct spelling & grammar together, in one pass over the whole text** — a seq2seq transformer
4. **Recover per-word detections** by diffing the model's input against its output, and label each change spelling / grammar / other
5. **Color-code** those changes on the original text
6. **Normalize** each word
7. **Stem** each word
8. **Lemmatize** each word

## Why a seq2seq transformer for step 3

A word-by-word or adjacent-word-pair checker (the previous version of this
notebook) can only catch errors it has a specific rule for, and can't use
information from anywhere else in the sentence. A **sequence-to-sequence
transformer** reads the *entire* input as one sequence and generates a
corrected version of the *entire* output as one sequence — so a correction
at position 3 can be informed by a word at position 30, and spelling and
grammar are corrected together in the same pass rather than as two separate
mechanisms bolted together.

**Model used: [`CAMeL-Lab/arabart-qalb14-gec-ged-13`](https://huggingface.co/CAMeL-Lab/arabart-qalb14-gec-ged-13
)** — an AraBART (BART-architecture, encoder-decoder) model fine-tuned for
Arabic GEC on the QALB-2014 dataset by the CAMeL Lab at NYU Abu Dhabi (the
same group behind CAMeL Tools, used elsewhere in this notebook), published at
EMNLP 2023, MIT licensed. Found by searching Hugging Face and GitHub for
Arabic GEC models — see the Notes section at the end for the other options
considered and why this one was picked.

**Two honest caveats about this model, upfront:**

1. In the paper, this GEC model gets an *auxiliary* per-word error-tag input
   from a companion error-detection model, via a **custom fork of
   `transformers`** the authors provide. This notebook deliberately does
   **not** install that fork — it uses the model with the plain, standard
   `transformers` library, which measurably simplifies setup at some cost to
   correction quality (you're getting the fine-tuned seq2seq model itself,
   just without its optional auxiliary signal). See the Notes section for
   the full explanation and a link if you want the complete paper pipeline.
2. **This specific model call could not be executed in the sandbox this
   notebook was written in** — outbound network access there is restricted
   to package registries (PyPI, GitHub, etc.) and doesn't include
   huggingface.co, so the model weights themselves couldn't be downloaded to
   test against. Every other cell in this notebook (including the diffing,
   classification, and rendering logic that sits on top of this step) *was*
   executed end-to-end and confirmed working. This step is written
   defensively (wrapped in error handling that reports what went wrong
   rather than crashing) precisely because of that gap — please open an
   issue or adjust the model name if it needs a fix on your end.

## Tools used (all researched on Hugging Face / GitHub / PyPI)

| Task | Tool | Source |
|---|---|---|
| Spelling + grammar correction (whole-text, seq2seq) | **AraBART GEC** (`CAMeL-Lab/arabart-qalb14-gec-ged-13`) | huggingface.co/CAMeL-Lab/arabart-qalb14-gec-ged-13 |
| Morphological preprocessing + lemmatization + POS | **CAMeL Tools** (NYU Abu Dhabi CAMeL Lab) | github.com/CAMeL-Lab/camel_tools |
| Classifying each detected change as spelling vs. grammar | CAMeL Tools morphological analyzer (word has zero valid analyses -> spelling) | (same) |
| Light stemming / root extraction | **Tashaphyne** | github.com/linuxscout/tashaphyne |
| Extra normalization helpers | **PyArabic** | github.com/linuxscout/pyarabic |
| Language identification (secondary check) | **langdetect** | pip: `langdetect` |

## How to use this notebook

Run the cells top to bottom once. Setup downloads ~130MB of CAMeL Tools data
plus the AraBART GEC model (~550MB). After that, every step has its own cell
and its own printed output. **"Full pipeline"** ties every step into one
function, and **"Try it with your own text"** runs that function on anything
you type without re-running setup. A GPU runtime (Runtime > Change runtime
type > GPU) makes generation faster but isn't required.

> **License note:** the AraBART GEC model, CAMeL Tools, PyArabic, and
> symspellpy-family tools are MIT/permissive. Tashaphyne is GPL-3.0 — worth
> knowing if you plan to redistribute this as part of a closed-source
> product.

## 0. Setup

In [1]:
# camel-tools already pulls in torch + transformers as dependencies; sentencepiece and
# protobuf are added explicitly because the AraBART tokenizer used in Step 3 needs them
# and does NOT install them automatically (a good way to find this out for yourself is
# a NameError/ImportError the first time you tokenize -- adding them upfront avoids that).
!pip install -q camel-tools tashaphyne langdetect pandas transformers torch sentencepiece protobuf

import sys
print(f"Python: {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("camel-tools needs Python 3.11+. In Colab: Runtime > Change runtime type.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.5/251.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires di

In [2]:
# Only the two CAMeL Tools data packages this notebook uses (morphology analysis DB +
# MLE disambiguation model for MSA) -- see the equivalent cell in the previous version of
# this notebook for why these are named explicitly instead of using the `light` bundle.
!camel_data -i morphology-db-msa-r13
!camel_data -i disambig-mle-calima-msa-r13

The following packages will be installed: 'morphology-db-msa-r13'
Extracting package 'morphology-db-msa-r13': 100% 40.5M/40.5M [00:00<00:00, 377MB/s]
The following packages will be installed: 'disambig-mle-calima-msa-r13'
Extracting package 'disambig-mle-calima-msa-r13': 100% 88.7M/88.7M [00:00<00:00, 106MB/s]


In [28]:
!pip install symspellpy
!wget -q -O ar_50k.txt "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2016/ar/ar_50k.txt"
!wc -l ar_50k.txt

50000 ar_50k.txt


In [3]:
import re
import html
import difflib
import pandas as pd
import torch
from IPython.display import display, HTML

from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar, normalize_alef_maksura_ar, normalize_teh_marbuta_ar,
)
import pyarabic.araby as araby
from tashaphyne.stemming import ArabicLightStemmer
from langdetect import detect_langs, DetectorFactory, LangDetectException
DetectorFactory.seed = 0  # deterministic langdetect results

print("Loading morphological analyzer (used to classify spelling vs. grammar changes) ...")
analyzer = Analyzer(MorphologyDB.builtin_db())

print("Loading MLE disambiguator (POS / gender / number / lemma / preprocessing) ...")
mle_disambiguator = MLEDisambiguator.pretrained()

light_stemmer = ArabicLightStemmer()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Ready. Using device: {device}")

Loading morphological analyzer (used to classify spelling vs. grammar changes) ...
Loading MLE disambiguator (POS / gender / number / lemma / preprocessing) ...
Ready. Using device: cpu


## Sample text

Same running example as before: a spelling typo (**المدرصة**), a
gender-agreement error (**الطالبة الجديد**), and a diacritized word
(**صباحاً**) to show normalization doing something. Edit `input_text` and
re-run the "Full pipeline" cell later to try something else.

In [26]:
input_text = "ذهبت الطالبة الجديد إلى المدرصة ضباحاً. هي تحبان القراءة كتيراً، ولديها كتاب جميلان تقرأه كل يوم."
print(input_text)

ذهبت الطالبة الجديد إلى المدرصة ضباحاً. هي تحبان القراءة كتيراً، ولديها كتاب جميلان تقرأه كل يوم.


## Step 1 — Verify the text is Arabic

Two independent signals: a deterministic **Unicode-range ratio** (primary —
what fraction of letters are in an Arabic Unicode block) and **`langdetect`**
(secondary, informational — a statistical language-ID model that's
noticeably less reliable on short or mixed-script text, since it can confuse
Arabic with other Arabic-script languages like Urdu or Persian).

In [20]:
ARABIC_RE = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]')

def arabic_char_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    arabic_letters = [c for c in letters if ARABIC_RE.match(c)]
    return len(arabic_letters) / len(letters)

def check_is_arabic(text, ratio_threshold=0.6):
    ratio = arabic_char_ratio(text)
    try:
        top = detect_langs(text)[0]
        ld_lang, ld_conf = top.lang, round(top.prob, 3)
    except LangDetectException:
        ld_lang, ld_conf = None, None
    return {
        'is_arabic': ratio >= ratio_threshold,
        'arabic_char_ratio': round(ratio, 3),
        'langdetect_lang': ld_lang,
        'langdetect_confidence': ld_conf,
    }

for sample in [input_text, "This is plain English.", "مرحبا hello مزيج نص"]:
    result = check_is_arabic(sample)
    label = "Arabic" if result['is_arabic'] else "NOT Arabic"
    print(f"[{label:10s}] ratio={result['arabic_char_ratio']:.0%}  "
          f"langdetect={result['langdetect_lang']} ({result['langdetect_confidence']})  "
          f"-> {sample[:40]!r}")

arabic_check = check_is_arabic(input_text)
assert arabic_check['is_arabic'], "Input does not look like Arabic -- stopping."

[Arabic    ] ratio=100%  langdetect=ar (1.0)  -> 'ذهبت الطالبة الجديد إلى المدرصة ضباحاً. '
[NOT Arabic] ratio=0%  langdetect=en (1.0)  -> 'This is plain English.'
[Arabic    ] ratio=69%  langdetect=fa (0.571)  -> 'مرحبا hello مزيج نص'


## Step 2 — Tokenization

Splits text into words/punctuation, and defines `is_arabic_token`, used
later to skip punctuation/digits/non-Arabic words. It checks for Arabic base
letters rather than using `str.isalpha()`, because `isalpha()` returns
`False` for a word that has Arabic diacritics attached (they're Unicode
"combining marks", not "letters") — which would silently skip vocalized
words like **صباحاً** in every later step otherwise.

In [21]:
ARABIC_LETTERS_RE = re.compile(r'[\u0621-\u063A\u0641-\u064A\u066E\u066F\u0671-\u06D3\u06D5]')
LATIN_RE = re.compile(r'[A-Za-z]')

def is_arabic_token(tok):
    return bool(ARABIC_LETTERS_RE.search(tok)) and not LATIN_RE.search(tok) and not tok.isdigit()

def tokenize_arabic(text):
    return simple_word_tokenize(text)

tokens = tokenize_arabic(input_text)
print(tokens)
print(f"\n{len(tokens)} tokens, {sum(is_arabic_token(t) for t in tokens)} of which are Arabic words")

['ذهبت', 'الطالبة', 'الجديد', 'إلى', 'المدرصة', 'ضباحاً', '.', 'هي', 'تحبان', 'القراءة', 'كتيراً', '،', 'ولديها', 'كتاب', 'جميلان', 'تقرأه', 'كل', 'يوم', '.']

19 tokens, 16 of which are Arabic words


## Step 3 — Spelling & grammar correction, whole text at once (seq2seq transformer)

Loads the AraBART GEC model and runs it over the input as a single sequence
(splitting only if the text is long enough to risk exceeding the model's
length budget, in which case it's split on sentence boundaries and each
sentence still gets full-sentence context).

Before generation, each word is replaced with its dediacritized, disambiguated
form via CAMeL Tools' MLE disambiguator — an approximation of the heavier
BERT-based morphological preprocessing the model was actually fine-tuned
with (see Notes at the end). This keeps setup to one extra model instead of
two, at some cost to how closely it matches the paper's exact preprocessing.

For better results, I used the word based spelling correction first to catch the in word misspelling and correct it to be easy on the AraBART GEC model to correct the errors

In [30]:
import re
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from symspellpy import SymSpell, Verbosity

analyzer = Analyzer(MorphologyDB.builtin_db())

ARABIC_LETTERS_RE = re.compile(r'[\u0621-\u063A\u0641-\u064A\u066E\u066F\u0671-\u06D3\u06D5]')
LATIN_RE = re.compile(r'[A-Za-z]')
def is_arabic_token(tok):
    return bool(ARABIC_LETTERS_RE.search(tok)) and not LATIN_RE.search(tok) and not tok.isdigit()
def tokenize_arabic(text):
    return simple_word_tokenize(text)

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary('ar_50k.txt', term_index=0, count_index=1, separator=' ', encoding='utf-8')

def correct_spelling_words(tokens):
    fixed = []
    changed = {}
    for tok in tokens:
        if not is_arabic_token(tok) or analyzer.analyze(tok):
            fixed.append(tok)
            continue
        suggestions = sym_spell.lookup(tok, Verbosity.CLOSEST, max_edit_distance=2)
        valid = [s.term for s in suggestions if analyzer.analyze(s.term)]
        best = valid[0] if valid else (suggestions[0].term if suggestions else None)
        if best:
            fixed.append(best)
            changed[tok] = best
        else:
            fixed.append(tok)
    return fixed, changed

# both example sentences used in the notebook

tokens = tokenize_arabic(input_text)
fixed, changed = correct_spelling_words(tokens)
print("IN: ", tokens)
print("OUT:", fixed)
print("changed:", changed)
input_text = ' '.join(fixed)
print("corrected:", input_text)

IN:  ['ذهبت', 'الطالبة', 'الجديد', 'إلى', 'المدرصة', 'ضباحاً', '.', 'هي', 'تحبان', 'القراءة', 'كتيراً', '،', 'ولديها', 'كتاب', 'جميلان', 'تقرأه', 'كل', 'يوم', '.']
OUT: ['ذهبت', 'الطالبة', 'الجديد', 'إلى', 'المدرسة', 'صباحا', '.', 'هي', 'تحبان', 'القراءة', 'كثيرا', '،', 'ولديها', 'كتاب', 'جميلان', 'تقرأه', 'كل', 'يوم', '.']
changed: {'المدرصة': 'المدرسة', 'ضباحاً': 'صباحا', 'كتيراً': 'كثيرا'}
corrected: ذهبت الطالبة الجديد إلى المدرسة صباحا . هي تحبان القراءة كثيرا ، ولديها كتاب جميلان تقرأه كل يوم .


In [31]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GEC_MODEL_NAME = 'CAMeL-Lab/arabart-qalb14-gec-ged-13'

try:
    print(f"Loading {GEC_MODEL_NAME} ...")
    gec_tokenizer = AutoTokenizer.from_pretrained(GEC_MODEL_NAME)
    gec_model = AutoModelForSeq2SeqLM.from_pretrained(GEC_MODEL_NAME).to(device)
    gec_model.eval()
    gec_model_available = True
    print(f"Loaded on {device}.")
except Exception as e:
    gec_model_available = False
    print(f"Could not load the GEC model ({type(e).__name__}: {e})")
    print("Falling back to leaving text unchanged in this step -- see the markdown "
          "cell above for why this call couldn't be pre-tested, and check that the "
          "model name above is still correct on huggingface.co if this persists.")

SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!\u061F])\s+')

def morph_preprocess(tokens):
    # Approximates the paper's BERT-based morphological preprocessing using the
    # MLE disambiguator already loaded above: each word -> its dediacritized,
    # disambiguated surface form.
    disambiguated = mle_disambiguator.disambiguate(tokens)
    out = []
    for dw in disambiguated:
        if dw.analyses:
            out.append(dediac_ar(dw.analyses[0].analysis['diac']))
        else:
            out.append(dw.word)
    return out

def gec_generate_piece(text_piece):
    piece_tokens = tokenize_arabic(text_piece)
    preprocessed = ' '.join(morph_preprocess(piece_tokens))
    inputs = gec_tokenizer([preprocessed], return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        generated = gec_model.generate(
            **inputs,
            num_beams=5,
            max_length=min(512, inputs['input_ids'].shape[1] + 64),
            no_repeat_ngram_size=0,
            early_stopping=False,
        )
    return gec_tokenizer.batch_decode(generated, skip_special_tokens=True,
                                       clean_up_tokenization_spaces=False)[0]

def correct_with_seq2seq(text):
    if not gec_model_available:
        return text
    try:
        token_count = len(gec_tokenizer(text)['input_ids'])
        pieces = [text] if token_count <= 400 else \
            [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
        corrected_pieces = [gec_generate_piece(p) for p in pieces]
        return ' '.join(corrected_pieces)
    except Exception as e:
        print(f"Generation failed ({type(e).__name__}: {e}); leaving text unchanged for this step.")
        return text

corrected_text = correct_with_seq2seq(input_text)
print("Input:    ", input_text)
print("Corrected:", corrected_text)

Loading CAMeL-Lab/arabart-qalb14-gec-ged-13 ...


Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

[transformers] MBartForConditionalGeneration LOAD REPORT from: CAMeL-Lab/arabart-qalb14-gec-ged-13
Key                                 | Status     |  | 
------------------------------------+------------+--+-
model.encoder.embed_ged_tags.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded on cpu.
Input:     ذهبت الطالبة الجديد إلى المدرسة صباحا . هي تحبان القراءة كثيرا ، ولديها كتاب جميلان تقرأه كل يوم .
Corrected: ذهبت الطالبة الجديدة إلى المدرسة صباحا . هي تحب القراءة كثيرا ، ولديها كتاب جميل تقرؤه كل يوم .


## Step 4 — Recovering per-word detections from the correction

A seq2seq model doesn't label *which* words were wrong — it just outputs a
corrected sequence. To get back per-word detections (needed for color-coding
and for a per-word table), the original and corrected token sequences are
aligned with a sequence diff (`difflib`), and every non-matching span is
treated as a detected-and-corrected error.

Each single-word replacement is then labeled using the same morphological
**Analyzer** from before: if the *original* word has zero valid analyses, it
was a spelling problem; if it's a valid word that the model still changed,
it was almost certainly a grammar/agreement/word-choice problem. Multi-word
spans and insertions/deletions (missing or extra words) are labeled "other"
rather than forced into that binary.

In [12]:
def find_and_classify_edits(original_tokens, corrected_tokens):
    sm = difflib.SequenceMatcher(None, original_tokens, corrected_tokens, autojunk=False)
    edits = []
    insertions = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':
            continue
        orig_span = original_tokens[i1:i2]
        corr_span = corrected_tokens[j1:j2]
        if tag == 'insert':
            insertions.append(' '.join(corr_span))
            continue
        if tag == 'replace' and len(orig_span) == 1 and len(corr_span) == 1:
            category = 'spelling' if len(analyzer.analyze(orig_span[0])) == 0 else 'grammar'
        else:
            category = 'other'
        edits.append({
            'tag': tag,
            'orig_positions': list(range(i1, i2)),
            'original': ' '.join(orig_span),
            'correction': ' '.join(corr_span) if corr_span else '(removed)',
            'category': category,
        })
    return edits, insertions

corrected_tokens = tokenize_arabic(corrected_text)
edits, insertions = find_and_classify_edits(tokens, corrected_tokens)

print(f"{len(edits)} change(s) detected:")
for e in edits:
    print(f"  [{e['category']:8s}] '{e['original']}'  ->  '{e['correction']}'")
if insertions:
    print(f"\nWord(s) the model added that weren't in the original: {insertions}")

5 change(s) detected:
  [grammar ] 'الجديد'  ->  'الجديدة'
  [other   ] 'المدرصة ضباحاً'  ->  'المدرسة ضباحا'
  [grammar ] 'تحبان'  ->  'تحب'
  [spelling] 'كتيراً'  ->  'كثيرا'
  [other   ] 'جميلان تقرأه'  ->  'جميل تقرؤه'


## Step 5 — Color-code the detected changes

Same idea as before: the original sentence rendered with each changed word
highlighted — red for spelling, orange for grammar, purple for other
(multi-word or structural) edits — with a hover tooltip showing the model's
suggested correction.

In [13]:
CATEGORY_STYLE = {
    'spelling': ('#ffd6d6', '#d33'),
    'grammar':  ('#ffe8c2', '#e08a00'),
    'other':    ('#e6d9ff', '#7a4fd6'),
}

def render_highlighted_html(tokens, edits):
    position_info = {}
    for e in edits:
        for p in e['orig_positions']:
            position_info[p] = e

    spans = []
    for i, tok in enumerate(tokens):
        safe_tok = html.escape(tok)
        if i in position_info:
            e = position_info[i]
            bg, border = CATEGORY_STYLE[e['category']]
            title = f"{e['category']}: suggested '{e['correction']}'"
            spans.append(
                f'<span style="background:{bg};border-bottom:2px solid {border};'
                f'border-radius:3px;padding:1px 4px;" title="{html.escape(title)}">{safe_tok}</span>'
            )
        else:
            spans.append(safe_tok)

    legend = ''.join(
        f'<span style="background:{bg};border-bottom:2px solid {border};padding:1px 6px;'
        f'border-radius:3px;margin-inline-end:8px;">{cat}</span>'
        for cat, (bg, border) in CATEGORY_STYLE.items()
    )
    body = (
        f'<div dir="rtl" style="font-size:22px;line-height:2.4;'
        f'font-family:\'Traditional Arabic\',Tahoma,Arial,sans-serif;">{" ".join(spans)}</div>'
    )
    display(HTML(f'<div style="margin-bottom:8px;font-family:sans-serif;font-size:13px;color:#444;">{legend}</div>{body}'))

render_highlighted_html(tokens, edits)

## Step 6 — Normalization

Canonicalizes each word's orthography (runs on the corrected text from Step
3): strip elongation (tatweel), strip diacritics, unify alef variants
(أ إ آ ٱ → ا), alef maksura → yeh (ى → ي), teh marbuta → heh (ة → ه). This
intentionally runs *after* correction, on already-correct words.

In [14]:
def normalize_word(word):
    w = araby.strip_tatweel(word)
    w = dediac_ar(w)
    w = normalize_alef_ar(w)
    w = normalize_alef_maksura_ar(w)
    w = normalize_teh_marbuta_ar(w)
    return w

def normalize_text(tokens):
    return [normalize_word(t) if is_arabic_token(t) else t for t in tokens]

normalized_tokens = normalize_text(corrected_tokens)

for before, after in zip(corrected_tokens, normalized_tokens):
    marker = "  <- changed" if before != after else ""
    print(f"  {before:12s} -> {after}{marker}")

  ذهبت         -> ذهبت
  الطالبة      -> الطالبه  <- changed
  الجديدة      -> الجديده  <- changed
  إلى          -> الي  <- changed
  المدرسة      -> المدرسه  <- changed
  ضباحا        -> ضباحا
  .            -> .
  هي           -> هي
  تحب          -> تحب
  القراءة      -> القراءه  <- changed
  كثيرا        -> كثيرا
  ،            -> ،
  ولديها       -> ولديها
  كتاب         -> كتاب
  جميل         -> جميل
  تقرؤه        -> تقرؤه
  كل           -> كل
  يوم          -> يوم
  .            -> .


## Step 7 — Stemming

**Tashaphyne** light-stems each word (strips known Arabic prefixes/suffixes
via a finite-state automaton) and also returns its own guess at the
triliteral/quadriliteral root. Light stemming is a surface-pattern
technique — it doesn't validate against a dictionary, so it can occasionally
over-strip a word whose first letters happen to resemble a common prefix. A
known, published characteristic of light stemmers generally.

In [15]:
def stem_text(tokens):
    stems, roots = [], []
    for t in tokens:
        if is_arabic_token(t):
            light_stemmer.light_stem(t)
            stems.append(light_stemmer.get_stem())
            roots.append(light_stemmer.get_root())
        else:
            stems.append(t)
            roots.append(t)
    return stems, roots

stems, roots = stem_text(corrected_tokens)

for word, stem, root in zip(corrected_tokens, stems, roots):
    if is_arabic_token(word):
        print(f"  {word:12s} stem={stem:10s} root={root}")

  ذهبت         stem=ذهب        root=ذهب
  الطالبة      stem=طالب       root=طلب
  الجديدة      stem=جديد       root=جدد
  إلى          stem=إلى        root=إلى
  المدرسة      stem=مدرس       root=درس
  ضباحا        stem=ضباح       root=ضبح
  هي           stem=هي         root=هي
  تحب          stem=حب         root=حبب
  القراءة      stem=قراء       root=قرء
  كثيرا        stem=ثيرا       root=كثر
  ولديها       stem=لدى        root=لدى
  كتاب         stem=تاب        root=توب
  جميل         stem=جميل       root=جمل
  تقرؤه        stem=قرؤ        root=قرء
  كل           stem=كل         root=كل
  يوم          stem=وم         root=يوم


## Step 8 — Lemma analysis

The lemma (dictionary/citation form) comes from the MLE disambiguator's top
analysis for each word in the corrected text — the same tool used for
preprocessing in Step 3. CAMeL lemmas carry a homograph index (e.g.
`جَدِيد_1`), stripped here for display, along with diacritics.

Genuinely different from stemming: the lemma is a real, valid dictionary
word from morphological analysis; a stem is just "whatever's left after
cutting known affixes" and isn't necessarily a word on its own — see `تحب`
below: stem `حب` (truncated), lemma `أحب` (the actual verb "to love").

In [16]:
def analyze_words_in_context(tokens):
    disambiguated = mle_disambiguator.disambiguate(tokens)
    infos = []
    for i, dw in enumerate(disambiguated):
        if dw.analyses:
            a = dw.analyses[0].analysis
            infos.append({'index': i, 'word': dw.word, 'lex': a.get('lex'), 'pos': a.get('pos')})
        else:
            infos.append({'index': i, 'word': dw.word, 'lex': None, 'pos': None})
    return infos

def lemmatize_text(tokens):
    infos = analyze_words_in_context(tokens)
    lemmas = []
    for info in infos:
        if info['lex']:
            lemmas.append(dediac_ar(re.sub(r'_\d+$', '', info['lex'])))
        else:
            lemmas.append(info['word'])
    return lemmas

lemmas = lemmatize_text(corrected_tokens)

for word, stem, lemma in zip(corrected_tokens, stems, lemmas):
    if is_arabic_token(word):
        print(f"  {word:12s} stem={stem:10s} lemma={lemma}")

  ذهبت         stem=ذهب        lemma=ذهب
  الطالبة      stem=طالب       lemma=طالب
  الجديدة      stem=جديد       lemma=جديد
  إلى          stem=إلى        lemma=إلى
  المدرسة      stem=مدرس       lemma=مدرسة
  ضباحا        stem=ضباح       lemma=ضباحا
  هي           stem=هي         lemma=هي
  تحب          stem=حب         lemma=أحب
  القراءة      stem=قراء       lemma=قراءة
  كثيرا        stem=ثيرا       lemma=كثير
  ولديها       stem=لدى        lemma=لدى
  كتاب         stem=تاب        lemma=كتاب
  جميل         stem=جميل       lemma=جميل
  تقرؤه        stem=قرؤ        lemma=تقرؤه
  كل           stem=كل         lemma=كل
  يوم          stem=وم         lemma=يوم


## Full pipeline

Everything above, wrapped into one function that runs start to finish on any
text and returns a structured report, a per-word table, and the color-coded
before/after view.

In [17]:
def run_arabic_pipeline(text, show_output=True):
    report = {'input_text': text}

    arabic_check = check_is_arabic(text)
    report['arabic_check'] = arabic_check
    if not arabic_check['is_arabic']:
        if show_output:
            print(f"This does not look like Arabic text (Arabic character ratio "
                  f"{arabic_check['arabic_char_ratio']:.0%}). Stopping.")
        return report

    tokens = tokenize_arabic(text)
    corrected_text = correct_with_seq2seq(text)
    corrected_tokens = tokenize_arabic(corrected_text)
    edits, insertions = find_and_classify_edits(tokens, corrected_tokens)

    if show_output:
        print("Detected changes, highlighted on the original text:")
        render_highlighted_html(tokens, edits)

    normalized_tokens = normalize_text(corrected_tokens)
    stems, roots = stem_text(corrected_tokens)
    lemmas = lemmatize_text(corrected_tokens)

    # Note: corrected_tokens can be a different length than the original tokens now
    # (the seq2seq model can insert/delete words, unlike the old position-preserving
    # rule-based corrector), so this table describes the corrected text on its own
    # rather than trying to align it back to original-token positions.
    table = pd.DataFrame({
        'corrected_word': corrected_tokens,
        'normalized': normalized_tokens,
        'stem': stems,
        'root': roots,
        'lemma': lemmas,
    })

    report.update({
        'tokens': tokens,
        'corrected_text': corrected_text,
        'corrected_tokens': corrected_tokens,
        'edits': edits,
        'insertions': insertions,
        'table': table,
    })

    if show_output:
        print(f"\nCorrected text:\n{corrected_text}")
        print("\nPer-word breakdown (of the corrected text):")
        display(table)

    return report

_ = run_arabic_pipeline(input_text)

Detected changes, highlighted on the original text:



Corrected text:
ذهبت الطالبة الجديدة إلى المدرسة ضباحا . هي تحب القراءة كثيرا ، ولديها كتاب جميل تقرؤه كل يوم .

Per-word breakdown (of the corrected text):


,corrected_word,normalized,stem,root,lemma
0,ذهبت,ذهبت,ذهب,ذهب,ذهب
1,الطالبة,الطالبه,طالب,طلب,طالب
2,الجديدة,الجديده,جديد,جدد,جديد
3,إلى,الي,إلى,إلى,إلى
4,المدرسة,المدرسه,مدرس,درس,مدرسة
5,ضباحا,ضباحا,ضباح,ضبح,ضباحا
6,.,.,.,.,.
7,هي,هي,هي,هي,هي
8,تحب,تحب,حب,حبب,أحب
9,القراءة,القراءه,قراء,قرء,قراءة


## Try it with your own text

Edit `my_text` and re-run this one cell — no need to re-run Setup. The
seeded example flips the agreement error the other way round (masculine
noun + feminine adjective) and uses a fresh typo.

In [18]:
my_text = "اشترى الطالب قلماً جديدة من مكتزة قريبة."
_ = run_arabic_pipeline(my_text)

Detected changes, highlighted on the original text:



Corrected text:
اشترى الطالب قلما جديدة من مكتزة قريبة .

Per-word breakdown (of the corrected text):


,corrected_word,normalized,stem,root,lemma
0,اشترى,اشتري,شترى,شتر,ٱشترى
1,الطالب,الطالب,طالب,طلب,طالب
2,قلما,قلما,قلما,قلما,قلم
3,جديدة,جديده,جديد,جدد,جديد
4,من,من,من,من,من
5,مكتزة,مكتزه,مكتز,كتز,مكتزه
6,قريبة,قريبه,قريب,قرب,قريب
7,.,.,.,.,.


## Notes, scope, and known limitations

**On the GEC model and the custom-fork tradeoff:** the paper behind
`CAMeL-Lab/arabart-qalb14-gec-ged-13` feeds the GEC model an auxiliary
per-word error-tag sequence produced by a companion token-classification
model (`CAMeL-Lab/camelbert-msa-qalb14-ged-13`), via a custom fork of
`transformers` that adds a `ged_tags` argument to `.generate()`. That fork
isn't installed here, so this notebook uses the model through the plain,
standard `transformers` library and skips that auxiliary signal entirely —
`from_pretrained` loads the fine-tuned encoder-decoder weights fine (any
extra weights specific to the auxiliary-tag machinery are simply not there
to load, since the standard model class doesn't define them) and
`.generate()` runs as an ordinary seq2seq call. That's a deliberate
simplicity-over-completeness tradeoff, likely at some cost to correction
quality relative to the paper's full pipeline. If you want the complete
setup: https://github.com/CAMeL-Lab/arabic-gec.

**On testing:** every other computed cell in this notebook was executed
end-to-end (including a real Jupyter-kernel run of the previous, rule-based
version) before being delivered. This specific model call could not be —
the development sandbox's network access doesn't extend to huggingface.co.
If this cell errors for you, the model name is the first thing to check on
huggingface.co, followed by your `transformers` version.

**On the morphological-preprocessing approximation:** the model card says
it was fine-tuned on text preprocessed with a BERT-based disambiguator
(`camel_tools.disambig.bert.BERTUnfactoredDisambiguator`). This notebook
approximates that with the lighter MLE disambiguator already loaded for
Step 8, to avoid a second, heavier neural model. Likely close enough to be
useful, not guaranteed identical to what the model saw in training.

**Other Arabic GEC models considered:**
- `CAMeL-Lab/arabart-qalb15-gec-ged-13` / `CAMeL-Lab/arabart-zaebuc-gec-ged-13` — same family, fine-tuned on QALB-2015 / ZAEBUC instead of QALB-2014. Same custom-fork caveat applies; swap the model name to try them.
- AraT5 (`UBC-NLP/AraT5v2-base-1024`) fine-tuned for GEC — reported in at least one paper (ArbESC+, using a `"grammar:"` input prefix) but no publicly released fine-tuned checkpoint was found on Hugging Face at the time of writing, so it's not usable directly.
- Generic multilingual GEC (mT5/mBART-based, e.g. the "gT5" approach covering Arabic among 7 languages) — promising in the literature, but no confirmed public Hugging Face checkpoint was found either.

**Other things worth knowing:**
- **Detection is entirely a byproduct of correction** — if the seq2seq model misses an error, it won't be flagged; if it "fixes" something that wasn't broken, that shows up as a false positive. There's no independent detector cross-checking it anymore (the previous version's rule-based agreement checker played that role; this version trades that safety net for whole-text context).
- **The spelling/grammar split is a heuristic**, applied after the fact to single-word substitutions only; multi-word edits and insertions/deletions are labeled "other" rather than forced into that binary.
- **Light stemming (Tashaphyne) can over-strip** words whose first letters coincidentally resemble a known prefix — a general trait of light stemmers, not specific to this word list.
- **The MLE disambiguator scores each word independently** (no full sentence-level neural context), so lemma/POS choice can occasionally land on a less-likely-but-plausible reading for ambiguous forms.

**Further reading:**
- Alhafni et al., "Advancements in Arabic Grammatical Error Detection and Correction: An Empirical Investigation" (EMNLP 2023): https://aclanthology.org/2023.emnlp-main.396
- CAMeL Tools docs: https://camel-tools.readthedocs.io
- QALB shared tasks (the datasets these models are trained/evaluated on): https://camel.abudhabi.nyu.edu/qalb-shared-task-2015/